# Step 1 — Data Quality Validation

Data checks for flight operations, written for an operational support analyst context.

**Scope:** Controlled loading, profiling, and validation only. No cleaning, modelling, or BI development.

In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

print("Imports loaded.")

Imports loaded.


## 01 — Configuration and Paths

In [2]:
NOTEBOOK_PATH = Path.cwd()


def find_project_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(5):
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
        p = p.parent
    raise RuntimeError("Project root not found. Run Jupyter from project root or keep /data and /notebooks in place.")

PROJECT_ROOT = find_project_root(NOTEBOOK_PATH)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FLIGHTS_CSV = RAW_DIR / "flights.csv"
AIRLINES_CSV = RAW_DIR / "airlines.csv"
AIRPORTS_CSV = RAW_DIR / "airports.csv"
CANCEL_CODES_CSV = RAW_DIR / "cancellation_codes.csv"

RUN_TS = datetime.now(timezone.utc).isoformat(timespec="seconds")

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RUN_TS: {RUN_TS}")
print(f"RAW_DIR exists: {RAW_DIR.exists()}")
print(f"PROCESSED_DIR exists: {PROCESSED_DIR.exists()}")
print(f"flights.csv exists: {FLIGHTS_CSV.exists()}")
if FLIGHTS_CSV.exists():
    print(f"flights.csv size (MB): {FLIGHTS_CSV.stat().st_size/1e6:.1f}")

PROJECT_ROOT: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio
RUN_TS: 2026-03-07T15:56:34+00:00
RAW_DIR exists: True
PROCESSED_DIR exists: True
flights.csv exists: True
flights.csv size (MB): 592.4


## 02 — Identity Definition (Composite Key)

In [3]:
KEY_COLS = [
    "YEAR", "MONTH", "DAY",
    "AIRLINE", "FLIGHT_NUMBER",
    "ORIGIN_AIRPORT", "DESTINATION_AIRPORT",
    "SCHEDULED_DEPARTURE"
]

print("KEY_COLS defined:", KEY_COLS)

KEY_COLS defined: ['YEAR', 'MONTH', 'DAY', 'AIRLINE', 'FLIGHT_NUMBER', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE']


## 03 — Controlled Load Strategy

In [4]:
# Columns required for Step 1 validation
USECOLS = [
    # Identity
    "YEAR", "MONTH", "DAY",
    "AIRLINE", "FLIGHT_NUMBER",
    "ORIGIN_AIRPORT", "DESTINATION_AIRPORT",
    "SCHEDULED_DEPARTURE",

    # Time logic
    "DEPARTURE_TIME",
    "SCHEDULED_ARRIVAL",
    "ARRIVAL_TIME",

    # Delays
    "DEPARTURE_DELAY",
    "ARRIVAL_DELAY",

    # Operational flags
    "CANCELLED",
    "DIVERTED",
    "CANCELLATION_REASON",
]

DTYPE_MAP = {
    # Identity
    "YEAR": "Int16",
    "MONTH": "Int8",
    "DAY": "Int8",
    "FLIGHT_NUMBER": "Int32",

    # Flags
    "CANCELLED": "Int8",
    "DIVERTED": "Int8",

    # Codes
    "AIRLINE": "category",
    "ORIGIN_AIRPORT": "category",
    "DESTINATION_AIRPORT": "category",
    "CANCELLATION_REASON": "category",

    # Delays
    "DEPARTURE_DELAY": "Int32",
    "ARRIVAL_DELAY": "Int32",

    # Times
    "SCHEDULED_DEPARTURE": "Int32",
    "DEPARTURE_TIME": "Int32",
    "SCHEDULED_ARRIVAL": "Int32",
    "ARRIVAL_TIME": "Int32",
}

print("USECOLS count:", len(USECOLS))
print("DTYPE_MAP configured.")

print("Reading flights.csv header only...")
flights_cols = pd.read_csv(FLIGHTS_CSV, nrows=0).columns.tolist()
print(f"Column count in file: {len(flights_cols)}")

USECOLS count: 16
DTYPE_MAP configured.
Reading flights.csv header only...
Column count in file: 31


In [5]:
print("Loading flights.csv with controlled schema...")

flights = pd.read_csv(
    FLIGHTS_CSV,
    usecols=USECOLS,
    dtype=DTYPE_MAP,
    low_memory=False
)

print("Load complete.")
print("Shape:", flights.shape)
mem_mb = flights.memory_usage(deep=True).sum() / 1e6
print(f"Approx memory usage: {mem_mb:.1f} MB")

Loading flights.csv with controlled schema...
Load complete.
Shape: (5819079, 16)
Approx memory usage: 302.6 MB


## 04 — Structural Profiling

In [6]:
print("Dtypes:")
print(flights.dtypes)

print("\nObject columns (expected none):")
print(flights.select_dtypes(include="object").columns.tolist())

print("\nTotal rows:", len(flights))
print("\nNull counts (absolute):")
print(flights.isna().sum())

print("\nNull percentages (%):")
null_pct = (flights.isna().mean() * 100).round(2)
print(null_pct)

Dtypes:
YEAR                      Int16
MONTH                      Int8
DAY                        Int8
AIRLINE                category
FLIGHT_NUMBER             Int32
ORIGIN_AIRPORT         category
DESTINATION_AIRPORT    category
SCHEDULED_DEPARTURE       Int32
DEPARTURE_TIME            Int32
DEPARTURE_DELAY           Int32
SCHEDULED_ARRIVAL         Int32
ARRIVAL_TIME              Int32
ARRIVAL_DELAY             Int32
DIVERTED                   Int8
CANCELLED                  Int8
CANCELLATION_REASON    category
dtype: object

Object columns (expected none):
[]

Total rows: 5819079

Null counts (absolute):
YEAR                         0
MONTH                        0
DAY                          0
AIRLINE                      0
FLIGHT_NUMBER                0
ORIGIN_AIRPORT               0
DESTINATION_AIRPORT          0
SCHEDULED_DEPARTURE          0
DEPARTURE_TIME           86153
DEPARTURE_DELAY          86153
SCHEDULED_ARRIVAL            0
ARRIVAL_TIME             92513
ARRIVAL_DELA

## 05 — Data Quality Framework Setup

In [7]:
DQ_RESULTS = []
ISSUE_SAMPLES = []

def add_dq_result(check_name, failed_mask, severity="HIGH", notes=""):
    total = len(flights)
    failed_count = int(failed_mask.sum())
    failed_pct = (failed_count / total) * 100

    DQ_RESULTS.append({
        "check_name": check_name,
        "severity": severity,
        "rows_checked": total,
        "failed_count": failed_count,
        "failed_pct": round(failed_pct, 6),
        "notes": notes
    })

def hhmm_to_minutes(series):
    hours = series // 100
    minutes = series % 100
    return (hours * 60 + minutes)

def unique_cols(cols):
    return list(dict.fromkeys(cols))

def add_issue_samples(issue_type, mask, cols, max_rows=200):
    mask_count = int(mask.sum())
    if mask_count == 0:
        return
    cols = unique_cols(cols)
    sample = flights.loc[mask, cols].head(max_rows).copy()
    sample.insert(0, "issue_type", issue_type)
    ISSUE_SAMPLES.append(sample)

print("Data quality framework initialised.")

Data quality framework initialised.


## 06 — Validation Checks

### 06.1 Cancellation Consistency

In [8]:
print("Checking cancellation consistency...")

cancelled = flights["CANCELLED"] == 1
has_reason = flights["CANCELLATION_REASON"].notna()

cancelled_count = int(cancelled.sum())
print(f"Cancelled flights: {cancelled_count:,}")
print(f"Cancelled %: {round(cancelled.mean() * 100, 3)}")

bad_missing_reason = cancelled & (~has_reason)
bad_reason_when_not_cancelled = (~cancelled) & has_reason

print(f"Cancelled but missing reason: {int(bad_missing_reason.sum()):,}")
print(f"Not cancelled but has reason: {int(bad_reason_when_not_cancelled.sum()):,}")

Checking cancellation consistency...
Cancelled flights: 89,884
Cancelled %: 1.545
Cancelled but missing reason: 0
Not cancelled but has reason: 0


### 06.2 Duplicate Identity Check (Composite Key)

In [9]:
print("Checking duplicate flight instances by composite key...")

dup_mask = flights.duplicated(subset=KEY_COLS, keep=False)
dup_count = int(dup_mask.sum())

add_dq_result(
    "duplicate_flight_instance_by_composite_key",
    dup_mask,
    severity="CRITICAL",
    notes=f"Composite key: {KEY_COLS}"
)

print(f"Duplicate rows (including all copies): {dup_count:,}")
print(f"Duplicate %: {round((dup_count / len(flights)) * 100, 6)}")
if dup_count > 0:
    sample = flights.loc[dup_mask, KEY_COLS].head(10)
    print("Sample duplicate keys (first 10 rows):")
    display(sample)
else:
    print("No duplicates detected by composite key.")

Checking duplicate flight instances by composite key...
Duplicate rows (including all copies): 0
Duplicate %: 0.0
No duplicates detected by composite key.


### 06.3 Delay Sanity Thresholds

In [10]:
print("Checking delay sanity thresholds...")

EARLY_LIMIT = -60
LATE_LIMIT = 1440

# Thresholds flag values likely outside normal reporting windows
dep_delay = flights["DEPARTURE_DELAY"]
arr_delay = flights["ARRIVAL_DELAY"]

bad_dep_too_early = dep_delay < EARLY_LIMIT
bad_dep_too_late = dep_delay > LATE_LIMIT
bad_arr_too_early = arr_delay < EARLY_LIMIT
bad_arr_too_late = arr_delay > LATE_LIMIT

print(f"DEPARTURE_DELAY < {EARLY_LIMIT}: {int(bad_dep_too_early.sum()):,}")
print(f"DEPARTURE_DELAY > {LATE_LIMIT}: {int(bad_dep_too_late.sum()):,}")
print(f"ARRIVAL_DELAY < {EARLY_LIMIT}: {int(bad_arr_too_early.sum()):,}")
print(f"ARRIVAL_DELAY > {LATE_LIMIT}: {int(bad_arr_too_late.sum()):,}")

add_dq_result("departure_delay_too_early", bad_dep_too_early, severity="MEDIUM", notes=f"< {EARLY_LIMIT} minutes")
add_dq_result("departure_delay_too_late", bad_dep_too_late, severity="HIGH", notes=f"> {LATE_LIMIT} minutes")
add_dq_result("arrival_delay_too_early", bad_arr_too_early, severity="MEDIUM", notes=f"< {EARLY_LIMIT} minutes")
add_dq_result("arrival_delay_too_late", bad_arr_too_late, severity="HIGH", notes=f"> {LATE_LIMIT} minutes")

print("Rows with extreme late delays (> 1440):")
ext_dep = flights["DEPARTURE_DELAY"] > LATE_LIMIT
ext_arr = flights["ARRIVAL_DELAY"] > LATE_LIMIT
print(f"Departure extreme count: {int(ext_dep.sum()):,}")
print(f"Arrival extreme count: {int(ext_arr.sum()):,}")
print(f"Overlap (both): {int((ext_dep & ext_arr).sum()):,}")

display(
    flights.loc[ext_dep, KEY_COLS + ["DEPARTURE_DELAY", "ARRIVAL_DELAY", "CANCELLED", "DIVERTED"]]
    .sort_values("DEPARTURE_DELAY", ascending=False)
    .head(10)
)

print("Extreme early arrivals (< -60):")
early_arr = flights["ARRIVAL_DELAY"] < EARLY_LIMIT
print(f"Count: {int(early_arr.sum()):,}")
display(
    flights.loc[early_arr, KEY_COLS + ["ARRIVAL_DELAY", "DEPARTURE_DELAY", "CANCELLED", "DIVERTED"]]
    .sort_values("ARRIVAL_DELAY")
    .head(10)
)

Checking delay sanity thresholds...
DEPARTURE_DELAY < -60: 3
DEPARTURE_DELAY > 1440: 32
ARRIVAL_DELAY < -60: 392
ARRIVAL_DELAY > 1440: 32
Rows with extreme late delays (> 1440):
Departure extreme count: 32
Arrival extreme count: 32
Overlap (both): 31


,YEAR,MONTH,DAY,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_DELAY,ARRIVAL_DELAY,CANCELLED,DIVERTED
337720,2015,1,23,AA,1322,BHM,DFW,700,1988,1971,0,0
3412085,2015,8,1,AA,96,RIC,DFW,709,1878,1898,0,0
4103531,2015,9,13,AA,1063,SAN,DFW,700,1670,1665,0,0
5810811,2015,12,31,AA,2214,ABQ,DFW,1041,1649,1636,0,0
5279939,2015,11,27,AA,2559,DTW,ORD,1027,1631,1638,0,0
3100911,2015,7,13,AA,1319,IND,LAX,1905,1625,1636,0,0
1278418,2015,3,24,AA,1279,OMA,DFW,1103,1609,1598,0,0
264495,2015,1,18,AA,224,LAS,LAX,1130,1604,1593,0,0
949876,2015,3,4,AA,270,HNL,LAX,828,1589,1576,0,0
886984,2015,2,28,AA,1312,STL,MIA,620,1587,1627,0,0


Extreme early arrivals (< -60):
Count: 392


,YEAR,MONTH,DAY,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,ARRIVAL_DELAY,DEPARTURE_DELAY,CANCELLED,DIVERTED
1467349,2015,4,4,AA,2483,PHX,DFW,2150,-87,-68,0,0
1093297,2015,3,12,US,693,HNL,PHX,2215,-87,-8,0,0
316347,2015,1,21,AS,11,EWR,SEA,1720,-82,-15,0,0
4535993,2015,10,10,VX,165,11618,12892,1230,-81,-7,0,0
1836251,2015,4,27,UA,1225,EWR,LAX,1907,-81,3,0,0
5136512,2015,11,17,AA,211,JFK,LAS,1700,-80,-13,0,0
3627775,2015,8,13,AA,60,DFW,MIA,2030,-80,-4,0,0
5645762,2015,12,20,AS,161,ADK,ANC,1715,-80,-82,0,0
4538501,2015,10,10,DL,420,12478,12892,1530,-79,0,0,0
561640,2015,2,6,DL,1104,HNL,SLC,2150,-79,-1,0,0


### 06.4 Cancelled vs Actual Time Behaviour

In [11]:
print("Checking cancelled flights against actual times...")

cancelled = flights["CANCELLED"] == 1
has_actual_dep = flights["DEPARTURE_TIME"].notna()
has_actual_arr = flights["ARRIVAL_TIME"].notna()
has_dep_delay = flights["DEPARTURE_DELAY"].notna()
has_arr_delay = flights["ARRIVAL_DELAY"].notna()

bad_cancel_dep_time = cancelled & has_actual_dep
bad_cancel_arr_time = cancelled & has_actual_arr
bad_cancel_dep_delay = cancelled & has_dep_delay
bad_cancel_arr_delay = cancelled & has_arr_delay

print(f"Cancelled but has DEPARTURE_TIME: {int(bad_cancel_dep_time.sum()):,}")
print(f"Cancelled but has ARRIVAL_TIME: {int(bad_cancel_arr_time.sum()):,}")
print(f"Cancelled but has DEPARTURE_DELAY: {int(bad_cancel_dep_delay.sum()):,}")
print(f"Cancelled but has ARRIVAL_DELAY: {int(bad_cancel_arr_delay.sum()):,}")

# Nuance: in this dataset, some cancelled flights can have departure time while arrival time is null
problem_rows = flights[
    (flights["CANCELLED"] == 1) &
    (flights["DEPARTURE_TIME"].notna())
]
print("Sample cancelled flights with departure time:")
display(
    problem_rows[
        KEY_COLS + [
            "DEPARTURE_TIME",
            "ARRIVAL_TIME",
            "DEPARTURE_DELAY",
            "ARRIVAL_DELAY",
            "DIVERTED",
            "CANCELLATION_REASON"
        ]
    ].head(10)
)

Checking cancelled flights against actual times...
Cancelled but has DEPARTURE_TIME: 3,731
Cancelled but has ARRIVAL_TIME: 0
Cancelled but has DEPARTURE_DELAY: 3,731
Cancelled but has ARRIVAL_DELAY: 0
Sample cancelled flights with departure time:


,YEAR,MONTH,DAY,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,ARRIVAL_TIME,DEPARTURE_DELAY,ARRIVAL_DELAY,DIVERTED,CANCELLATION_REASON
4391,2015,1,1,MQ,2758,ROW,DFW,1110,1229,<NA>,79,<NA>,0,B
4766,2015,1,1,EV,4503,IAH,BRO,1134,1129,<NA>,-5,<NA>,0,B
7441,2015,1,1,EV,4264,STL,IAH,1423,1628,<NA>,125,<NA>,0,C
8516,2015,1,1,EV,4707,IAH,HRL,1530,1537,<NA>,7,<NA>,0,B
10454,2015,1,1,MQ,3240,PIA,DFW,1735,1835,<NA>,60,<NA>,0,A
13560,2015,1,1,EV,5968,DEN,ELP,2157,2246,<NA>,49,<NA>,0,C
16999,2015,1,2,AA,1222,TUS,DFW,815,821,<NA>,6,<NA>,0,A
17716,2015,1,2,MQ,3128,ORD,CVG,855,855,<NA>,0,<NA>,0,A
25124,2015,1,2,EV,2530,DFW,BTR,1615,1846,<NA>,151,<NA>,0,A
27988,2015,1,2,UA,344,IAH,LGA,1905,2040,<NA>,95,<NA>,0,A


### 06.5 Time vs Delay Consistency (Midnight-Aware)

In [12]:
print("Checking time vs recorded departure delay...")

sched_dep_min = hhmm_to_minutes(flights["SCHEDULED_DEPARTURE"])
actual_dep_min = hhmm_to_minutes(flights["DEPARTURE_TIME"])

valid_rows = (
    flights["SCHEDULED_DEPARTURE"].notna() &
    flights["DEPARTURE_TIME"].notna() &
    (flights["CANCELLED"] == 0)
)

sched_dep_min = sched_dep_min[valid_rows]
actual_dep_min = actual_dep_min[valid_rows]
recorded_delay = flights.loc[valid_rows, "DEPARTURE_DELAY"]

calculated_delay = actual_dep_min - sched_dep_min

# Midnight rollover handling: very negative differences are treated as next-day departures
calculated_delay = calculated_delay.where(
    calculated_delay >= -720,
    calculated_delay + 1440
)

TOLERANCE = 5
delay_diff = (calculated_delay - recorded_delay).abs()
bad_delay_mismatch = delay_diff > TOLERANCE

print(f"Checked rows: {len(delay_diff):,}")
print(f"Mismatches: {int(bad_delay_mismatch.sum()):,}")
print(f"Mismatch %: {round((bad_delay_mismatch.mean() * 100), 4)}")

print("Sample delay mismatches:")
mismatch_rows = flights.loc[valid_rows].loc[bad_delay_mismatch]
display(
    mismatch_rows[
        KEY_COLS + [
            "DEPARTURE_TIME",
            "DEPARTURE_DELAY"
        ]
    ].head(10)
)

Checking time vs recorded departure delay...
Checked rows: 5,729,195
Mismatches: 865
Mismatch %: 0.0151
Sample delay mismatches:


,YEAR,MONTH,DAY,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY
0,2015,1,1,AS,98,ANC,SEA,5,2354,-11
6371,2015,1,1,AA,2470,BOS,DFW,1315,905,1190
13858,2015,1,1,DL,1840,LIH,LAX,2319,1339,860
13950,2015,1,2,AA,2400,LAX,DFW,5,2359,-6
13951,2015,1,2,AS,98,ANC,SEA,5,2353,-12
22654,2015,1,2,DL,1743,ECP,ATL,1349,454,905
28381,2015,1,2,AA,1302,RDU,DFW,1930,1015,885
29432,2015,1,2,AA,1072,DFW,TPA,2040,913,753
35353,2015,1,3,AA,1677,MEM,DFW,1010,910,1380
39834,2015,1,3,DL,401,SJU,JFK,1457,613,916


## 07 — Export Outputs

In [13]:
print("Building dq_summary.csv...")

dq_summary = pd.DataFrame(DQ_RESULTS).sort_values(
    ["severity", "failed_count"],
    ascending=[True, False]
)

display(dq_summary.head(10))

dq_summary_path = PROCESSED_DIR / "dq_summary.csv"
dq_summary.to_csv(dq_summary_path, index=False)
print(f"Saved: {dq_summary_path}")

Building dq_summary.csv...


,check_name,severity,rows_checked,failed_count,failed_pct,notes
0,duplicate_flight_instance_by_composite_key,CRITICAL,5819079,0,0.000000,"Composite key: ['YEAR', 'MONTH', 'DAY', 'AIRLI..."
2,departure_delay_too_late,HIGH,5819079,32,0.000550,> 1440 minutes
4,arrival_delay_too_late,HIGH,5819079,32,0.000550,> 1440 minutes
3,arrival_delay_too_early,MEDIUM,5819079,392,0.006736,< -60 minutes
1,departure_delay_too_early,MEDIUM,5819079,3,0.000052,< -60 minutes


Saved: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio\data\processed\dq_summary.csv


In [14]:
print("Building issues_samples.csv...")

EARLY_LIMIT = -60
LATE_LIMIT = 1440

bad_dep_too_late = flights["DEPARTURE_DELAY"] > LATE_LIMIT
bad_arr_too_late = flights["ARRIVAL_DELAY"] > LATE_LIMIT
bad_arr_too_early = flights["ARRIVAL_DELAY"] < EARLY_LIMIT

sched_dep_min = hhmm_to_minutes(flights["SCHEDULED_DEPARTURE"])
actual_dep_min = hhmm_to_minutes(flights["DEPARTURE_TIME"])
valid_rows = (
    flights["SCHEDULED_DEPARTURE"].notna() &
    flights["DEPARTURE_TIME"].notna() &
    (flights["CANCELLED"] == 0)
)

calculated_delay = (actual_dep_min - sched_dep_min)
calculated_delay = calculated_delay.where(calculated_delay >= -720, calculated_delay + 1440)

TOLERANCE = 5
delay_diff = (calculated_delay[valid_rows] - flights.loc[valid_rows, "DEPARTURE_DELAY"]).abs()
bad_delay_mismatch = delay_diff > TOLERANCE

mismatch_full_mask = pd.Series(False, index=flights.index)
mismatch_full_mask.loc[bad_delay_mismatch.index] = True

base_cols = KEY_COLS + ["CANCELLED", "DIVERTED"]

add_issue_samples(
    "departure_delay_extreme_late",
    bad_dep_too_late,
    base_cols + ["DEPARTURE_TIME", "DEPARTURE_DELAY"]
)

add_issue_samples(
    "arrival_delay_extreme_late",
    bad_arr_too_late,
    base_cols + ["SCHEDULED_ARRIVAL", "ARRIVAL_TIME", "ARRIVAL_DELAY"]
)

add_issue_samples(
    "arrival_delay_extreme_early",
    bad_arr_too_early,
    base_cols + ["SCHEDULED_ARRIVAL", "ARRIVAL_TIME", "ARRIVAL_DELAY"]
)

add_issue_samples(
    "departure_delay_time_mismatch",
    mismatch_full_mask,
    base_cols + ["DEPARTURE_TIME", "DEPARTURE_DELAY"]
)

issues_samples = pd.concat(ISSUE_SAMPLES, ignore_index=True) if ISSUE_SAMPLES else pd.DataFrame()
issues_path = PROCESSED_DIR / "issues_samples.csv"
issues_samples.to_csv(issues_path, index=False)

print(f"Saved: {issues_path}")
print(f"issues_samples rows: {len(issues_samples):,}")
display(issues_samples.head(10))

Building issues_samples.csv...
Saved: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio\data\processed\issues_samples.csv
issues_samples rows: 464


,issue_type,YEAR,MONTH,DAY,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,CANCELLED,DIVERTED,DEPARTURE_TIME,DEPARTURE_DELAY,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY
0,departure_delay_extreme_late,2015,1,11,AA,1595,AUS,DFW,650,0,0,700,1450,<NA>,<NA>,<NA>
1,departure_delay_extreme_late,2015,1,18,AA,224,LAS,LAX,1130,0,0,1414,1604,<NA>,<NA>,<NA>
2,departure_delay_extreme_late,2015,1,21,AA,2385,JAX,DFW,1223,0,0,1240,1457,<NA>,<NA>,<NA>
3,departure_delay_extreme_late,2015,1,23,AA,1322,BHM,DFW,700,0,0,1608,1988,<NA>,<NA>,<NA>
4,departure_delay_extreme_late,2015,1,27,AA,1242,FAT,DFW,659,0,0,850,1551,<NA>,<NA>,<NA>
5,departure_delay_extreme_late,2015,2,22,AA,1080,EGE,ORD,1415,0,0,1432,1457,<NA>,<NA>,<NA>
6,departure_delay_extreme_late,2015,2,28,AA,1312,STL,MIA,620,0,0,847,1587,<NA>,<NA>,<NA>
7,departure_delay_extreme_late,2015,3,4,AA,270,HNL,LAX,828,0,0,1057,1589,<NA>,<NA>,<NA>
8,departure_delay_extreme_late,2015,3,10,AA,1594,SAT,DFW,850,0,0,1047,1557,<NA>,<NA>,<NA>
9,departure_delay_extreme_late,2015,3,24,AA,1279,OMA,DFW,1103,0,0,1352,1609,<NA>,<NA>,<NA>


## 08 — Step 1 Summary

Step 1 covers controlled loading, structural profiling, and validation checks before building reports.

**Generated outputs:**
- `data/processed/dq_summary.csv`
- `data/processed/issues_samples.csv`